# Qwen 3.6 Plus Cookbook

**Model:** `qwen-3.6-plus`  \n
**Latest update:** March 2026  \n
**Provider:** OpenRouter (OpenAI-compatible API)

This notebook shows practical patterns for Qwen 3.6 Plus via OpenRouter using the OpenAI Chat Completions API.

If your OpenRouter model slug differs from the default, update the `MODEL` value in the setup cell before running examples.

**Table of Contents:**
0. Setup and installation
1. Basic usage with provider SDK
2. Framework integration
3. Building agents
4. Advanced applications
5. Model-specific section: multilingual + long-context workflow

## 0. Setup and installation

In [1]:
import subprocess
import sys

packages = [
    "openai>=1.0.0",
    "pydantic>=2.0.0",
    "python-dotenv>=1.0.0",
]

cmd = [sys.executable, "-m", "pip", "install", "-q", *packages]
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    print("Installed core packages")
else:
    print("Package installation failed")
    if result.stderr:
        print(result.stderr.strip())

Installed core packages


In [ ]:
import os
from openai import OpenAI

# Define the model slug once and reuse it in all cells.
MODEL = os.getenv("OPENROUTER_MODEL", "qwen/qwen3.6-plus-preview:free")

# OpenRouter setup.
# https://openrouter.ai/docs/api-reference/overview
BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
API_KEY = "your api key here"

if not API_KEY:
    print("Warning: OPENROUTER_API_KEY is not set. Set environment variables before running API calls.")

client = OpenAI(
    api_key=API_KEY or "DUMMY_KEY",
    base_url=BASE_URL,
    default_headers={
        "HTTP-Referer": os.getenv("OPENROUTER_SITE_URL", "https://localhost"),
        "X-Title": os.getenv("OPENROUTER_APP_NAME", "qwen-3.6-plus-cookbook"),
    },
)
print(f"MODEL = {MODEL}")
print(f"BASE_URL = {BASE_URL}")

MODEL = qwen/qwen3.6-plus-preview:free
BASE_URL = https://openrouter.ai/api/v1


## 1. Basic usage with provider SDK

This section demonstrates `client.chat.completions.create` for text generation.

In [21]:
def basic_chat(prompt: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a concise coding assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content

# Example call (requires valid OPENROUTER_API_KEY):
# print(basic_chat("Write a Python function to deduplicate a list while preserving order."))

In [22]:
import asyncio

async def run_parallel_prompts(prompts: list[str]) -> list[str]:
    """Run multiple prompts concurrently using async threads around sync SDK calls."""

    async def one(prompt: str) -> str:
        return await asyncio.to_thread(basic_chat, prompt)

    tasks = [one(p) for p in prompts]
    return await asyncio.gather(*tasks)

async def main():
    prompts = [
        "Summarize what retrieval augmented generation is in two lines.",
        "List three production risks in agent systems.",
        "Give one SQL indexing tip for chat logs.",
    ]
    outputs = await run_parallel_prompts(prompts)
    for i, out in enumerate(outputs, start=1):
        print(f"Response {i}:\n{out}\n")

# await main()

## 2. Framework integration

This section demonstrates `ChatOpenAI` from `langchain-openai` with a Qwen-compatible endpoint.

In [23]:
# Optional framework package: install on-demand if missing
import subprocess
import sys

try:
    from langchain_openai import ChatOpenAI
except ModuleNotFoundError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "langchain-openai"],
        check=False,
    )
    from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=MODEL,
    api_key=API_KEY or "DUMMY_KEY",
    base_url=BASE_URL,
    temperature=0.1,
)

print("LangChain ChatOpenAI client created")
# Example call
# print(llm.invoke("Give a short CI/CD checklist for Python apps").content)

LangChain ChatOpenAI client created


## 3. Building agents

This section demonstrates tool-calling payload format with `tools` in Chat Completions.

In [24]:
import json
from datetime import datetime, timezone

def get_utc_time() -> str:
    return datetime.now(timezone.utc).isoformat()

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_utc_time",
            "description": "Returns the current UTC timestamp",
            "parameters": {
                "type": "object",
                "properties": {},
            },
        },
    }
]

messages = [
    {"role": "system", "content": "You are a scheduler assistant."},
    {"role": "user", "content": "What time is it in UTC now?"},
]

# First request: model may decide to call a tool.
try:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )

    msg = resp.choices[0].message
    tool_calls = msg.tool_calls or []

    if tool_calls:
        messages.append(msg.model_dump())
        for tc in tool_calls:
            if tc.function.name == "get_utc_time":
                tool_result = get_utc_time()
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps({"utc_time": tool_result}),
                })

        final_resp = client.chat.completions.create(model=MODEL, messages=messages)
        print(final_resp.choices[0].message.content)
    else:
        print(msg.content)
except Exception as e:
    print("Tool-calling example skipped due to API/model issue:")
    print(str(e))

It is currently 12:53:48 UTC on March 31, 2026.


## 4. Advanced applications

This section demonstrates structured JSON output and robust parsing.

In [27]:
from pydantic import AliasChoices, BaseModel, Field, ValidationError

class ActionItems(BaseModel):
    summary: str
    items: list[str] = Field(validation_alias=AliasChoices("items", "action_items"))

prompt = "Read this update and return JSON with summary and action items: We shipped OAuth login, need telemetry dashboards, and should backfill user audit logs."

try:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0,
    )

    raw = resp.choices[0].message.content
    print("Raw JSON:", raw)

    try:
        parsed = ActionItems.model_validate_json(raw)
        print("\nValidated object:")
        print(parsed.model_dump())
    except ValidationError as e:
        print("Validation failed:")
        print(e)
except Exception as e:
    print("Structured output example skipped due to API/model issue:")
    print(str(e))

Raw JSON: {
  "summary": "OAuth login has been successfully shipped. The team now needs to set up telemetry dashboards and backfill historical user audit logs.",
  "action_items": [
    "Create and configure telemetry dashboards",
    "Backfill user audit logs"
  ]
}

Validated object:
{'summary': 'OAuth login has been successfully shipped. The team now needs to set up telemetry dashboards and backfill historical user audit logs.', 'items': ['Create and configure telemetry dashboards', 'Backfill user audit logs']}


## 5. Model-specific section: multilingual + long-context workflow

Qwen models are commonly used for multilingual tasks and long documents. This pattern chunks a large input and produces a final synthesis.

In [26]:
def chunk_text(text: str, chunk_size: int = 4000) -> list[str]:
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

def summarize_chunks(text: str) -> str:
    parts = chunk_text(text)
    partial_summaries = []

    for idx, part in enumerate(parts, start=1):
        summary = basic_chat(
            f"Summarize chunk {idx}/{len(parts)} in English and Chinese.\n\n{part}"
        )
        partial_summaries.append(summary)

    combined = "\n\n".join(partial_summaries)
    final = basic_chat(
        "Merge the following bilingual summaries into one concise bilingual executive brief:\n\n" + combined
    )
    return final

# Example
# long_doc = "... your long input text ..."
# print(summarize_chunks(long_doc))

## Footer

Checklist before production use:
- Verify your provider model string for Qwen 3.6 Plus.
- Confirm max context window and pricing from official provider docs.
- Add retries, timeout, and request IDs in your API wrapper.
- Log prompt and tool traces with sensitive fields redacted.